In [ ]:
import random
import numpy as np
from datasets import load_dataset, get_dataset_config_names
from PIL import Image
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# 💥 학습 목표: 군사 이미지(Military Imagery)에서 사물 탐지(Object Detection)의 개념을 익힙니다.
# 💥 데이터셋명: llama-farm/military-labeled-yolo
# 💥 데이터셋의 의미: 공공 영역(Public Domain)의 군사 관련 이미지에 다양한 군사 장비(탱크, 비행기, 병사 등)를 YOLO 포맷으로 라벨링 한 데이터입니다.
# 💥 데이터셋의 용도: 대규모 컴퓨터 비전 모델(YOLO, Faster R-CNN 등)을 훈련시키기 위한 전형적인 '객체 탐지' 데이터셋입니다.
# -----------------------------------------------------------------------------

# --- 설정 상수 ---
DATASET_NAME = "llama-farm/military-labeled-yolo"
TARGET_SPLIT = 'train'
SAMPLE_COUNT = 5 # 초보자 실습을 위해 상위 5개 샘플만 사용합니다!

print("================================================================")
print("🤖 튜터: 안녕하세요! 군사 이미지를 이용한 객체 탐지 실습에 오신 것을 환영합니다! 💪")
print("지금부터 실제 데이터셋을 로드하며 AI 데이터 전처리 과정을 함께 살펴볼게요.")
print("================================================================")

# 1. 데이터셋 설정 확인 (Rule 21)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"\n[✅ 1단계] 사용 가능한 Config 목록 확인: {configs}")
    
    selected_config = configs[0]
except Exception as e:
    print(f"\n[ℹ️] 해당 데이터셋은 기본(default) 설정만 사용합니다. ({e})")
    selected_config = None


# 2. 데이터 로딩 시도 (Rule 2, 3, 18)
dataset = None
try:
    print("\n[🚀 2단계] Streaming 모드로 데이터셋 로드를 시도합니다 (빠른 처리 방식)...")
    dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=True)
    print("🎉 성공적으로 Streaming 모드로 데이터셋을 로드했습니다! 메모리 효율적이에요.")

except Exception as e:
    # Streaming 실패 시 (가끔 발생할 수 있음) 일반 모드로 폴백
    print(f"\n[🚨 경고!] Streaming 로딩에 실패했습니다. 일반 모드(Non-streaming)로 전환합니다. ({e})")
    try:
        dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=False)
        print("✅ 일반 모드(Non-streaming)로 데이터셋 로딩 성공!")
    except Exception as final_e:
        print(f"❌ 치명적인 오류: 데이터셋을 로드할 수 없습니다. {final_e}")
        exit()


# 3. 샘플링 준비 (Rule 9, 6)
print(f"\n[🔬 3단계] 학습을 위해 전체 데이터 중 상위 {SAMPLE_COUNT}개의 샘플을 추출하겠습니다.")

# dataset에 .take() 메서드가 있는지 확인 (Rule 9)
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset) 처리
    print("▶ Streaming 모드를 사용 중이므로, .take()를 이용해 반복자(iterator)를 준비합니다.")
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset) 처리
    print("▶ 일반 데이터셋을 사용 중이므로, .take()를 이용해 리스트를 준비합니다.")
    # list(dataset.take(SAMPLE_COUNT)) 패턴 사용 (Rule 16)
    sampled_dataset = list(dataset.take(SAMPLE_COUNT))
    sampled_dataset_iterator = iter(sampled_dataset)


# 4. 실습 데이터 분석 및 시각화 (Rule 4, 1-1)

print("\n" + "="*80)
print("🖼️ [4단계] ✨ 가상 전장 스캐너 분석 시작! ✨ (Object Detection 분석)")
print("="*80)

# 분석 결과를 저장할 변수
detected_object_counts = {}
sample_count_processed = 0

# 데이터 분석 루프
# Iterator를 사용하므로 next()를 사용하여 순회합니다.
print("👉 추출된 각 샘플을 순회하며, 이미지 크기와 감지된 객체(Detection) 정보를 분석합니다.")
try:
    while True:
        sample_data = next(sampled_dataset_iterator)
        sample_count_processed += 1
        
        # sample_data 내부의 핵심 정보 추출
        # 실제 YOLO 데이터셋은 이미지와 라벨을 포함하는 복잡한 구조를 가집니다.
        # 여기서는 객체 탐지의 '결과'와 '메타 정보'에 초점을 맞춥니다.
        
        # (Rule 14 방지) 간결하고 명확한 print 사용
        print(f"\n--- [샘플 {sample_count_processed}/{SAMPLE_COUNT}] 분석 시작 ---")
        
        # 1. 이미지 정보 분석
        # 실제 이미지 필드가 어떤 형태로 로드되었는지에 따라 처리 방식이 달라집니다.
        # 여기서는 PIL Image 혹은 NumPy Array 형태를 가정하고, .shape 또는 .size를 확인합니다.
        image_data = sample_data.get('image')
        if image_data is not None:
            if hasattr(image_data, 'shape'):
                img_shape = image_data.shape
            elif hasattr(image_data, 'size'):
                # PIL Image object 처리 (Rule 17)
                img_shape = image_data.size
            else:
                img_shape = "Unknown Shape"
            print(f"🔍 이미지 크기 (Dimensions): {img_shape}")
            
            # 간단한 시각화 (실제 이미지를 불러오는 예시)
            try:
                # 만약 이미지 데이터가 PIL 객체라면 바로 사용 가능합니다.
                if isinstance(image_data, Image.Image):
                    img_to_display = image_data
                else:
                    # NumPy 배열인 경우 PIL로 변환 시도
                    img_to_display = Image.fromarray(image_data)
                
                plt.figure(figsize=(6, 6))
                plt.imshow(img_to_display)
                plt.title("Sample Image View")
                plt.axis('off')
                plt.show()
                
            except Exception as e:
                print(f"⚠️ 이미지 시각화에 실패했습니다. (데이터 타입 문제일 수 있습니다. Error: {e})")
                
        # 2. 객체 탐지 결과 분석 (가장 창의적이고 중요한 부분)
        # 라벨 데이터는 일반적으로 YOLO 바운딩 박스 좌표와 클래스 ID로 구성됩니다.
        labels = sample_data.get('labels', [])
        
        if labels:
            print(f"📦 탐지된 객체 개수: {len(labels)}개")
            
            # 라벨 데이터에서 클래스 이름 추출 및 통계 분석
            # 여기서는 'class_name'이라는 메타 필드가 라벨에 포함되어 있다고 가정합니다.
            unique_classes = []
            for label in labels:
                # 가상의 클래스 이름을 추출하는 로직 (실제 필드에 맞게 수정 필요)
                # 실제 데이터셋 구조를 모르므로, 이 필드는 분석 예시로만 사용됩니다.
                if isinstance(label, dict) and 'class_name' in label:
                     class_name = label['class_name']
                elif isinstance(label, str):
                    class_name = label # 만약 문자열로 클래스명이 온다면
                else:
                    # 임시 디버그용 이름
                    class_name = "Unknown Object" 
                
                unique_classes.append(class_name)
                
                # 정량적 분석: 클래스별 개수 세기
                detected_object_counts[class_name] = detected_object_counts.get(class_name, 0) + 1
        else:
            print("✨ 이 샘플에서는 감지된 객체가 없습니다. 배경 이미지일 수 있습니다.")

finally:
    # 5. 최종 종합 분석 및 정리 (Rule 1-1, 4)
    
    print("\n" + "="*80)
    print("🏆 [결론] 전장 스캔 결과 종합 보고서")
    print("="*80)
    
    # 총 분석된 샘플 수 보고
    print(f"💡 분석 완료 샘플 수: 총 {sample_count_processed}개의 샘플 데이터를 분석했습니다.")

    # 객체 카운트 시각화/출력
    if detected_object_counts:
        print("\n⭐ Top 3 감지된 객체 종류 (가장 많이 발견된 상위 3가지):")
        # 가장 많이 발견된 상위 3개 클래스 추출
        sorted_classes = sorted(detected_object_counts.items(), key=lambda item: item[1], reverse=True)
        top_k = 3
        
        for i, (class_name, count) in enumerate(sorted_classes[:top_k]):
            print(f"    {i+1}. {class_name}: {count}회 감지")
            
        # 모든 클래스 목록 제공
        print("\n[ℹ️] 전체 감지된 유니크 클래스 목록:", list(detected_object_counts.keys()))
    else:
        print("❌ 데이터셋 메타 정보에는 명확한 객체 탐지 라벨이 포함되어 있지 않거나, 샘플에서 감지된 객체가 없습니다.")

print("\n================================================================")
print("🚀 실습 종료! 축하합니다! 다음 단계에서는 이 결과를 바탕으로 실제 YOLO 모델을 훈련시킬 수 있습니다.")
print("================================================================")